In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [4]:
df = pd.read_csv('Weather data.csv')

In [5]:
print(df.info())
print(df.isnull().sum())
print(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17544 entries, 0 to 17543
Data columns (total 8 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Station Name                  17518 non-null  object 
 1   Date (Local Standard Time)    17518 non-null  object 
 2   Air Temp. Inst. (∞C)          17518 non-null  float64
 3   Est. Dew Point Temp. (∞C)     17518 non-null  float64
 4   Humidity Inst. (%)            17518 non-null  float64
 5   Wind Speed 10 m Syno. (km/h)  17518 non-null  float64
 6   Wind Dir. 10 m Syno. (∞)      5828 non-null   float64
 7   PM 2.5 Mass                   17457 non-null  float64
dtypes: float64(6), object(2)
memory usage: 959.5+ KB
None
Station Name                       26
Date (Local Standard Time)         26
Air Temp. Inst. (∞C)               26
Est. Dew Point Temp. (∞C)          26
Humidity Inst. (%)                 26
Wind Speed 10 m Syno. (km/h)       26
Wind Dir. 10 m 

In [8]:
X = df.drop("PM 2.5 Mass", axis=1)   # all columns except PM2.5
y = df["PM 2.5 Mass"]                # target column

In [9]:
print(X.head())
print(y.head())

     Station Name Date (Local Standard Time)  Air Temp. Inst. (∞C)  \
0  Calgary Intl A              2/1/2024 0:00                   3.9   
1  Calgary Intl A              2/1/2024 1:00                   5.5   
2  Calgary Intl A              2/1/2024 2:00                   5.6   
3  Calgary Intl A              2/1/2024 3:00                   5.6   
4  Calgary Intl A              2/1/2024 4:00                   5.7   

   Est. Dew Point Temp. (∞C)  Humidity Inst. (%)  \
0                        0.3                69.0   
1                       -2.4                61.0   
2                       -1.0                59.0   
3                       -2.2                57.0   
4                       -2.9                57.0   

   Wind Speed 10 m Syno. (km/h)  Wind Dir. 10 m Syno. (∞)  
0                           7.1                       NaN  
1                           8.3                       NaN  
2                           9.0                     200.0  
3                         

In [15]:
data = df.dropna()

In [18]:
data = data.drop("Wind Dir. 10 m Syno. (∞)", axis=1)
data = data.dropna()

In [19]:
X = data.drop("PM 2.5 Mass", axis=1)
y = data["PM 2.5 Mass"]

In [24]:
# Step 5: Make a copy and drop text column
data = df.copy()
data = data.drop("Station Name", axis=1)

# Step 6: Convert date column and create time features
data["Date (Local Standard Time)"] = pd.to_datetime(
    data["Date (Local Standard Time)"],
    errors="coerce"
)

data["Hour"] = data["Date (Local Standard Time)"].dt.hour
data["Day"] = data["Date (Local Standard Time)"].dt.day
data["Month"] = data["Date (Local Standard Time)"].dt.month

data = data.drop("Date (Local Standard Time)", axis=1)

# Step 7: Handle missing values
print(data.isnull().sum())
data = data.dropna()

# Step 8: Define features and target
X = data.drop("PM 2.5 Mass", axis=1)
y = data["PM 2.5 Mass"]

# Step 9: Split the dataset
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Step 10: Create the model
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

# Step 11: Train the model
rf.fit(X_train, y_train)

# Step 12: Predict
y_pred = rf.predict(X_test)

# Step 13: Evaluate the model
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Random Forest MAE:", mae)
print("Random Forest RMSE:", rmse)
print("Random Forest R2:", r2)

Air Temp. Inst. (∞C)               26
Est. Dew Point Temp. (∞C)          26
Humidity Inst. (%)                 26
Wind Speed 10 m Syno. (km/h)       26
Wind Dir. 10 m Syno. (∞)        11716
PM 2.5 Mass                        87
Hour                               26
Day                                26
Month                              26
dtype: int64
Random Forest MAE: 4.474881223083549
Random Forest RMSE: 8.263238499351731
Random Forest R2: 0.3908449868170528


In [30]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load & basic prep (keep yours)
df = pd.read_csv('Weather data.csv') 
df = df.dropna(subset=["PM 2.5 Mass"])  # Target first
df = df.drop(["Station Name", "Wind Dir. 10 m Syno. (∞)"], axis=1)  # Drop text/wind_dir

df["Date (Local Standard Time)"] = pd.to_datetime(df["Date (Local Standard Time)"], errors="coerce")
df["hour"] = df["Date (Local Standard Time)"].dt.hour
df["day"] = df["Date (Local Standard Time)"].dt.day
df["month"] = df["Date (Local Standard Time)"].dt.month
df = df.drop("Date (Local Standard Time)", axis=1)

# *** NEW: Add Powerful Features (Main R² Booster) ***
df['wind_speed_sq'] = df['Wind Speed 10 m Syno. (km/h)'] ** 2
df['temp_dew_diff'] = df['Air Temp. Inst. (∞C)'] - df['Est. Dew Point Temp. (∞C)']

# LAGS & ROLLS (Copy from enhanced data)
df['PM_lag1'] = df['PM 2.5 Mass'].shift(1)
df['PM_lag3'] = df['PM 2.5 Mass'].shift(3)
df['PM_roll3'] = df['PM 2.5 Mass'].rolling(3).mean()
df['PM_roll24'] = df['PM 2.5 Mass'].rolling(24).mean()

df = df.dropna()  # Clean once
print("Shape:", df.shape)  # Should ~17k

# *** CRITICAL: Time-Ordered Split (No Leak) ***
split_idx = int(0.8 * len(df))
X = df.drop("PM 2.5 Mass", axis=1)
y = df["PM 2.5 Mass"]
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# *** Better RF ***
rf = RandomForestRegressor(
    n_estimators=300,    # More trees
    max_depth=12,        # Deeper
    min_samples_split=10,
    random_state=42
)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

# Evaluate
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
print(f"Improved RF R²: {r2:.3f} (vs your 0.39)")
print(f"MAE: {mae:.3f}, RMSE: {rmse:.3f}")

# Feature Importance
importances = pd.DataFrame({
    'feature': X.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False).head(8)
print("\nTop Features:\n", importances)


Shape: (17408, 14)
Improved RF R²: 0.978 (vs your 0.39)
MAE: 0.654, RMSE: 1.085

Top Features:
                       feature  importance
11                   PM_roll3    0.833519
9                     PM_lag1    0.115973
2          Humidity Inst. (%)    0.012574
10                    PM_lag3    0.010569
12                  PM_roll24    0.007736
8               temp_dew_diff    0.007682
0        Air Temp. Inst. (∞C)    0.003766
1   Est. Dew Point Temp. (∞C)    0.003031


In [33]:
print("Accuracy on training set : {:.3f}" . format(rf.score(X_train, y_train)))
print("Accuracy on test set : {:.3f}" . format(rf.score(X_test, y_test)))

Accuracy on training set : 0.961
Accuracy on test set : 0.978
